# 01 Inputs Validation
Question: Are macro and strategy inputs valid and aligned?

**Warning:** This notebook is descriptive only and must not be used for forecasting, signal generation, or trading decisions.

This notebook is the validation gatekeeper for all downstream notebooks. No returns-by-regime, transition, or risk analysis should be treated as in scope until the checks here pass cleanly.


In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing 'src' and 'data'.")

REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.data.loaders import load_regimes, load_strategy_returns, load_transitions
from src.data.validators import run_all_validations

regimes = load_regimes(REPO_ROOT / "data" / "macro" / "regimes_example.csv")
transitions = load_transitions(REPO_ROOT / "data" / "macro" / "transitions_example.csv")
strategy_returns = load_strategy_returns(REPO_ROOT / "data" / "strategies" / "strategy_returns_example.csv")

schema_checks = pd.DataFrame(
    [
        {
            "dataset": "regimes",
            "required_columns": "DATE, Growth_Regime, Inflation_Regime, Policy_Regime, Macro_Regime",
            "actual_columns": ", ".join(regimes.columns),
            "row_count": len(regimes),
        },
        {
            "dataset": "transitions",
            "required_columns": "transition_date, from, to",
            "actual_columns": ", ".join(transitions.columns),
            "row_count": len(transitions),
        },
        {
            "dataset": "strategy_returns",
            "required_columns": "DATE, strategy_id, returns",
            "actual_columns": ", ".join(strategy_returns.columns),
            "row_count": len(strategy_returns),
        },
    ]
)
display(schema_checks)

date_coverage = pd.DataFrame(
    [
        {
            "dataset": "regimes",
            "min_date": regimes["DATE"].min(),
            "max_date": regimes["DATE"].max(),
            "unique_dates": regimes["DATE"].nunique(),
        },
        {
            "dataset": "transitions",
            "min_date": transitions["transition_date"].min(),
            "max_date": transitions["transition_date"].max(),
            "unique_dates": transitions["transition_date"].nunique(),
        },
        {
            "dataset": "strategy_returns",
            "min_date": strategy_returns["DATE"].min(),
            "max_date": strategy_returns["DATE"].max(),
            "unique_dates": strategy_returns["DATE"].nunique(),
        },
    ]
)
display(date_coverage)

strategy_dates_not_in_macro = sorted(set(strategy_returns["DATE"]) - set(regimes["DATE"]))
coverage_diagnostics = pd.DataFrame(
    [
        {
            "check": "strategy dates covered by macro dates",
            "status": "PASS" if not strategy_dates_not_in_macro else "FAIL",
            "details": "all strategy dates covered" if not strategy_dates_not_in_macro else ", ".join(str(d.date()) for d in strategy_dates_not_in_macro[:5]),
        },
        {
            "check": "transition dates covered by macro dates",
            "status": "PASS" if set(transitions["transition_date"]).issubset(set(regimes["DATE"])) else "FAIL",
            "details": "all transition dates covered" if set(transitions["transition_date"]).issubset(set(regimes["DATE"])) else "transition date mismatch detected",
        },
    ]
)
display(coverage_diagnostics)

regime_counts = regimes["Macro_Regime"].value_counts(dropna=False).rename_axis("Macro_Regime").reset_index(name="count")
transition_counts = transitions.groupby(["from", "to"]).size().reset_index(name="count")
display(regime_counts)
display(transition_counts)

validation_status = "PASS"
validation_message = "All locked-interface validations passed. Downstream notebooks may proceed."
try:
    run_all_validations(regimes, transitions, strategy_returns)
except ValueError as exc:
    validation_status = "FAIL"
    validation_message = str(exc)
    raise
finally:
    status_frame = pd.DataFrame([{"validation_status": validation_status, "details": validation_message}])
    display(status_frame)
